# Indo EcoTourism — CBF + **User Feedback Weighting (UFW)**

This notebook builds a **Content-Based Filtering (CBF)** system for tourism destination recommendations and **adds lightweight personalization** through **User Feedback Weighting (UFW)**.

**Workflow overview:**
1. **Load & clean data** (`eco_place.csv`)
2. **Text preprocessing** → combine description, category, city → *TF-IDF*
3. **CBF ranker** + **UFW scoring**: `score = cos(query,item) + α · cos(centroid_like,item)`
4. **Offline evaluation**: Precision@1, Recall@10, F1@10, Latency  
5. **Justification** for using UFW based on the metrics above

## 1) Setup & Install
Environment preparation and library installation.

In [9]:
# !pip -q install pandas numpy scikit-learn scipy joblib


## 2) Imports & Paths
Import core libraries and set up the artifacts directory.

In [10]:
import os
import re
import json
import time
import random
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy import sparse
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import save_npz, csr_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
BASE_DIR = Path(".")
ART_DIR = BASE_DIR / "../artifacts"
ART_DIR.mkdir(exist_ok=True, parents=True)

print("Artifacts dir:", ART_DIR.resolve())

Artifacts dir: /Users/anpabelt/Documents/macbook/fyp/artifacts


## 3) Load Data

In [11]:
from IPython.display import display

DATA_CSV_PATH = [Path("../dataset/eco_place.csv"), Path("./eco_place.csv")]

if not any(p.exists() for p in DATA_CSV_PATH):
    raise FileNotFoundError(
        "eco_place.csv not found. Please upload/place the file in the notebook's working directory."
    )

df = pd.read_csv([p for p in DATA_CSV_PATH if p.exists()][0])
print("Loaded:", [p for p in DATA_CSV_PATH if p.exists()][0])
print("Shape:", df.shape)
display(df.head(3))

Loaded: ../dataset/eco_place.csv
Shape: (182, 13)


,place_id,place_name,place_description,category,city,price,rating,description_location,place_img,gallery_photo_img1,gallery_photo_img2,gallery_photo_img3,place_map
0,1,Taman Nasional Gunung Leuser,Taman Nasional Gunung Leuser adalah salah satu...,"Budaya,Taman Nasional",Aceh,Rp 25,4.5,"Barisan mountain range, Aceh 24653",https://i.ibb.co/JW6nC0bn/Taman-Nasional-Gunun...,https://i.ibb.co/0yntqm5C/Taman-Nasional-Gunun...,https://i.ibb.co/nsrS5XJN/Taman-Nasional-Gunun...,https://i.ibb.co/dw6kB5RB/Taman-Nasional-Gunun...,https://www.google.com/maps/search/Taman+Nasio...
1,2,Desa Wisata Munduk,Desa Wisata Munduk adalah sebuah desa di pegun...,Desa Wisata,Bali,Rp 10,4.5,"Munduk, Banjar, Kabupaten Buleleng, Bali",https://i.ibb.co/0p0YdcHN/Desa-Wisata-Munduk-p...,https://i.ibb.co/G6fvr5t/Desa-Wisata-Munduk-ga...,https://i.ibb.co/LdFdrHNY/Desa-Wisata-Munduk-g...,https://i.ibb.co/MyzF7nz9/Desa-Wisata-Munduk-g...,https://goo.gl/maps/LyeJ2mAeFGysTE9v9
2,3,Desa Wisata Penglipuran,Desa Wisata Penglipuran adalah sebuah desa wis...,"Budaya,Desa Wisata",Bali,Rp 25,4.8,"Jl. Penglipuran, Kubu, Kec. Bangli, Kabupaten ...",https://i.ibb.co/ZRCvs1tH/Desa-Wisata-Penglipu...,https://i.ibb.co/GQQ0ggTb/Desa-Wisata-Penglipu...,https://i.ibb.co/Kx73KwVj/Desa-Wisata-Penglipu...,https://i.ibb.co/Q7YPc1Gg/Desa-Wisata-Penglipu...,https://www.google.com/maps/search/Desa+Wisata...



## 4) Cleaning & Preprocessing


In [12]:
df = df.copy()

# Ensure required columns exist
required_cols = ["place_id", "place_name", "place_description", "category", "city", "price", "rating", "place_img", "place_map"]
for col in required_cols:
    if col not in df.columns:
        df[col] = np.nan

# Remove duplicates
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicates removed: {before - len(df)}")

# Parse IDR price (SIMPLIFIED)
def parse_price_idr(x):
    """Extract the first number from a price string."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    s = s.replace("rp", "").replace(".", "").replace(",", "")
    if "gratis" in s:
        return 0.0
    digits = re.findall(r"\d+", s)
    if digits:
        return float(digits[0])
    return np.nan

df["price"] = df["price"].apply(parse_price_idr)
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Indonesian stopwords (SIMPLIFIED - without IMPORTANT_WORDS)
STOPWORDS_ID = {
    "ada", "adalah", "agar", "akan", "atau", "banyak", "beberapa", "belum",
    "bila", "bisa", "bukan", "dalam", "dan", "dapat", "dari", "dengan", "di",
    "hanya", "harus", "hingga", "ini", "itu", "jika", "juga", "kami", "kamu",
    "karena", "ke", "kepada", "lain", "lalu", "lebih", "masih", "mereka",
    "namun", "nya", "oleh", "pada", "para", "saat", "saja", "sampai", "sangat",
    "sebagai", "sebuah", "seluruh", "semua", "serta", "setiap", "suatu", "sudah",
    "tanpa", "tapi", "tentang", "untuk", "yaitu", "yang"
}

def preprocess_text(text):
    """Text preprocessing: lowercase, tokenize, remove stopwords."""
    text = str(text).lower()
    tokens = re.findall(r"\w+", text)
    tokens = [t for t in tokens if t not in STOPWORDS_ID]
    return " ".join(tokens)

# Fill empty columns with empty strings
df["place_description"] = df["place_description"].fillna("").astype(str)
df["category"] = df["category"].fillna("").astype(str)
df["city"] = df["city"].fillna("").astype(str)
df["place_name"] = df["place_name"].fillna("").astype(str)

# Create combined column
df["combination"] = (df["place_description"] + " " + df["category"] + " " + df["city"]).apply(preprocess_text)

print("Nulls after cleaning:\n", df.isnull().sum())
df[["place_name", "category", "city", "combination"]].head(5)

Duplicates removed: 0
Nulls after cleaning:
 place_id                 0
place_name               0
place_description        0
category                 0
city                     0
price                    0
rating                   0
description_location     0
place_img                0
gallery_photo_img1       0
gallery_photo_img2       2
gallery_photo_img3      77
place_map                0
combination              0
dtype: int64


,place_name,category,city,combination
0,Taman Nasional Gunung Leuser,"Budaya,Taman Nasional",Aceh,taman nasional gunung leuser salah satu enam t...
1,Desa Wisata Munduk,Desa Wisata,Bali,desa wisata munduk desa pegunungan bali terken...
2,Desa Wisata Penglipuran,"Budaya,Desa Wisata",Bali,desa wisata penglipuran desa wisata terletak k...
3,Taman Nasional Bali Barat,"Taman Nasional,Cagar Alam",Bali,taman nasional bali barat kawasan konservasi a...
4,Bukit Jamur,Cagar Alam,Bandung,bukit jamur ciwidey satu sekian pesona wisata ...


## 5) Feature Extraction — TF‑IDF

In [13]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm="l2"
)
tfidf_matrix = vectorizer.fit_transform(df["combination"].fillna(""))
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (182, 1532)



## 6) Save Artifacts


In [14]:
items_cols = ["place_id", "place_name", "place_description", "category", "city", "price", "rating", "description_location", "place_img", "gallery_photo_img1", "gallery_photo_img2", "gallery_photo_img3", "place_map", "combination"]
items = df[items_cols].copy()
items.to_csv(ART_DIR / "items.csv", index=False)

# Save vectorizer
joblib.dump(vectorizer, ART_DIR / "vectorizer.joblib")

# Save TF-IDF matrix as numpy array
save_npz(ART_DIR / "tfidf_matrix.npz", csr_matrix(tfidf_matrix))
nbrs = NearestNeighbors(n_neighbors=50, metric="cosine", algorithm="brute")
nbrs.fit(tfidf_matrix)
joblib.dump(nbrs, ART_DIR / "nbrs_cosine.joblib")

# Metadata
meta = {
    "created_at": dt.datetime.utcnow().isoformat() + "Z",
    "n_items": int(items.shape[0]),
    "n_features": int(tfidf_matrix.shape[1]),
    "vectorizer": "sklearn TfidfVectorizer"
}
with open(ART_DIR / "metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("[OK] Artifacts saved:")
for p in sorted(ART_DIR.iterdir()):
    print(" -", p.name)

[OK] Artifacts saved:
 - items.csv
 - metadata.json
 - nbrs_cosine.joblib
 - tfidf_matrix.npz
 - vectorizer.joblib


/var/folders/rv/t7mzr88520bcvttrrmjtfftm0000gn/T/ipykernel_16684/2397746148.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": dt.datetime.utcnow().isoformat() + "Z",



## 7) Ranker: **CBF + User Feedback Weighting (UFW)**


In [15]:
# """
# Item score = cosine(query, item) + alpha * cosine(centroid_liked, item)

# - query: TF-IDF vector from user input
# - item: TF-IDF vector of the item
# - centroid_liked: mean vector of items liked by the user
# - alpha: feedback influence weight (default 0.6)
# """

# # Convert to dense array for computation
# TFIDF_DENSE = tfidf_matrix.toarray()

# def get_user_centroid(liked_indices):
#     """Compute the centroid (mean) of liked items."""
#     if not liked_indices:
#         return None
#     liked_vectors = TFIDF_DENSE[liked_indices]
#     centroid = liked_vectors.mean(axis=0).reshape(1, -1)
#     return centroid

# def rank_cbf_ufw(query, liked_indices, alpha=0.6, topk=10):
#     """
#     Ranking with CBF + User Feedback Weighting.

#     Args:
#         query: query string from the user
#         liked_indices: list of indices of items liked by the user
#         alpha: feedback weight (0-1)
#         topk: number of results to return

#     Returns:
#         list of item indices with the highest scores
#     """
#     # Transform query to TF-IDF vector
#     query_processed = preprocess_text(query)
#     query_vector = vectorizer.transform([query_processed])

#     # Compute similarity between query and all items
#     sim_query = cosine_similarity(query_vector, TFIDF_DENSE)[0]

#     # If no liked items, return pure CBF ranking
#     centroid = get_user_centroid(liked_indices)
#     if centroid is None:
#         top_indices = np.argsort(-sim_query)[:topk]
#         return top_indices.tolist()

#     # Compute similarity between centroid and all items
#     sim_centroid = cosine_similarity(centroid, TFIDF_DENSE)[0]

#     # Combine scores: CBF + alpha * UFW
#     scores = sim_query + alpha * sim_centroid

#     # Get top-k
#     top_indices = np.argsort(-scores)[:topk]
#     return top_indices.tolist()

In [16]:
# ---- 1) Load artifacts ----
ART_DIR = "artifacts"  # change if needed
items = pd.read_csv(f"../{ART_DIR}/items.csv")                 # preprocessed metadata
vectorizer = joblib.load(f"../{ART_DIR}/vectorizer.joblib")    # adjust filename if different
tfidf_matrix = sparse.load_npz(f"../{ART_DIR}/tfidf_matrix.npz")

"""
Item score = cosine(query, item) + alpha * cosine(centroid_liked, item)

- query: TF-IDF vector from user input
- item: TF-IDF vector of the item
- centroid_liked: mean vector of items liked by the user
- alpha: feedback influence weight (default 0.6)
"""

# Convert to dense array for computation
TFIDF_DENSE = tfidf_matrix.toarray()

# ---- 2) Ensure preprocess_text exists ----
try:
    preprocess_text
except NameError:
    import re
    def preprocess_text(text):
        text = str(text).lower()
        tokens = re.findall(r"\w+", text)
        return " ".join(tokens)

def get_user_centroid(liked_indices):
    """Compute the centroid (mean) of liked items."""
    if not liked_indices:
        return None
    liked_vectors = TFIDF_DENSE[liked_indices]
    centroid = liked_vectors.mean(axis=0).reshape(1, -1)
    return centroid

def rank_cbf_ufw(query, liked_indices, alpha=0.6, topk=10):
    """
    Ranking with CBF + User Feedback Weighting.

    Args:
        query: query string from the user
        liked_indices: list of indices of items liked by the user
        alpha: feedback weight (0-1)
        topk: number of results to return

    Returns:
        list of item indices with the highest scores
    """
    # Transform query to TF-IDF vector
    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])

    # Compute similarity between query and all items
    sim_query = cosine_similarity(query_vector, TFIDF_DENSE)[0]

    # If no liked items, return pure CBF ranking
    centroid = get_user_centroid(liked_indices)
    if centroid is None:
        top_indices = np.argsort(-sim_query)[:topk]
        return top_indices.tolist()

    # Compute similarity between centroid and all items
    sim_centroid = cosine_similarity(centroid, TFIDF_DENSE)[0]

    # Combine scores: CBF + alpha * UFW
    scores = sim_query + alpha * sim_centroid

    # Get top-k
    top_indices = np.argsort(-scores)[:topk]
    return top_indices.tolist()

# ---- 3) Score helper (for display) ----
def compute_scores(query, liked_indices=None, alpha=0.6):
    """
    Compute the same score vector used by rank_cbf_ufw, so we can display scores.
    Returns: scores array (len = number of items)
    """
    liked_indices = liked_indices or []

    query_processed = preprocess_text(query)
    query_vector = vectorizer.transform([query_processed])
    sim_query = cosine_similarity(query_vector, TFIDF_DENSE)[0]

    centroid = get_user_centroid(liked_indices)
    if centroid is None:
        return sim_query

    sim_centroid = cosine_similarity(centroid, TFIDF_DENSE)[0]
    return sim_query + alpha * sim_centroid


In [17]:
# ---- 4) Neat markdown display ----
def show_catalog(items, n=50):
    cols = ["place_name", "category", "city", "rating", "price"]
    cols = [c for c in cols if c in items.columns]

    df = items[cols].copy()
    df.insert(0, "idx", df.index)

    display(Markdown(f"### ECOTOURISM LIST (showing {min(n, len(df))} of {len(df)} items)"))
    display(Markdown("Choose liked indices from the **idx** column (comma-separated). Example: `10,25,50`"))
    display(df.head(n).style.hide(axis="index").format({"price": "{:.0f}"}))

def parse_liked_indices(user_input, n_items):
    liked = []
    for x in (user_input or "").split(","):
        x = x.strip()
        if x.isdigit():
            i = int(x)
            if 0 <= i < n_items:
                liked.append(i)
    seen = set()
    return [i for i in liked if not (i in seen or seen.add(i))]

def show_results(title, query, top_indices, scores, items):
    cols = ["place_name", "category", "city", "rating", "price"]
    cols = [c for c in cols if c in items.columns]

    df = items.loc[top_indices, cols].copy()
    df.insert(0, "idx", df.index)
    df.insert(1, "score", [float(scores[i]) for i in df["idx"]])

    display(Markdown(f"### {title}"))
    display(Markdown(f"**Query:** `{query}`"))

    display(
        df.style
          .hide(axis="index")
          .format({"score": "{:.4f}", "price": "{:.0f}"})
    )

# ---- 5) Two tests only ----
show_catalog(items, n=50)

query = "gunung"

# TEST 1: Basic search (no user feedback)
top1 = rank_cbf_ufw(query=query, liked_indices=[], alpha=0.6, topk=10)
scores1 = compute_scores(query=query, liked_indices=[], alpha=0.6)
show_results("TEST 1: Basic search (no user feedback)", query, top1, scores1, items)

# TEST 2: With user feedback (user-defined liked indices)
display(Markdown("### TEST 2: With user feedback (user-defined liked indices)"))
user_input = input("Enter liked indices (comma-separated), e.g. 10,25,50: ")
liked = parse_liked_indices(user_input, n_items=len(items))

display(Markdown(f"**Liked indices used:** `{liked}`"))
top2 = rank_cbf_ufw(query=query, liked_indices=liked, alpha=0.6, topk=10)
scores2 = compute_scores(query=query, liked_indices=liked, alpha=0.6)
show_results("TEST 2: With user feedback", query, top2, scores2, items)

### ECOTOURISM LIST (showing 50 of 182 items)

Choose liked indices from the **idx** column (comma-separated). Example: `10,25,50`

idx,place_name,category,city,rating,price
0,Taman Nasional Gunung Leuser,"Budaya,Taman Nasional",Aceh,4.500000,25
1,Desa Wisata Munduk,Desa Wisata,Bali,4.500000,10
2,Desa Wisata Penglipuran,"Budaya,Desa Wisata",Bali,4.800000,25
3,Taman Nasional Bali Barat,"Taman Nasional,Cagar Alam",Bali,4.500000,15
4,Bukit Jamur,Cagar Alam,Bandung,4.200000,12
5,Bukit Moko,Cagar Alam,Bandung,4.500000,25
6,Curug Bugbrug,Cagar Alam,Bandung,4.300000,8
7,Curug Cilengkrang,Cagar Alam,Bandung,4.000000,8
8,Curug Cimahi,Cagar Alam,Bandung,4.100000,20
9,Curug Cipanas,Cagar Alam,Bandung,4.000000,25


### TEST 1: Basic search (no user feedback)

**Query:** `gunung`

...

### TEST 2: With user feedback (user-defined liked indices)

**Liked indices used:** `[6, 7, 8]`

### TEST 2: With user feedback

**Query:** `gunung`

idx,score,place_name,category,city,rating,price
6,0.4215,Curug Bugbrug,Cagar Alam,Bandung,4.300000,8
7,0.4168,Curug Cilengkrang,Cagar Alam,Bandung,4.000000,8
8,0.4095,Curug Cimahi,Cagar Alam,Bandung,4.100000,20
17,0.3991,Gunung Manglayang,"Cagar Alam,Taman Nasional",Bandung,4.500000,8
172,0.3043,Puncak Gunung Api Purba - Nglanggeran,Cagar Alam,Yogyakarta,4.700000,10
16,0.3034,Gunung Lalakon,"Cagar Alam,Taman Nasional",Bandung,4.800000,0
135,0.2999,Bunker Kaliadem Merapi,Cagar Alam,Yogyakarta,4.500000,3
81,0.2916,Taman Nasional Gunung Rinjani,"Taman Nasional,Cagar Alam",Lombok,4.700000,10
87,0.2812,Taman Nasional Bromo Tengger Semeru,"Budaya,Taman Nasional",Malang,4.800000,30
48,0.2313,Taman Wisata Alam Gunung Pancar,Cagar Alam,Bogor,4.100000,20


## 8) Offline Evaluation

In [18]:
from IPython.display import display, Markdown

K_EVAL = 10

def build_query(row):
    """Build a synthetic query from item metadata."""
    cat = str(row.get("category", "")).split(",")[0].strip()
    city = str(row.get("city", "")).strip()
    name = str(row.get("place_name", "")).strip()
    parts = [p for p in [cat, city, name] if p]
    return " ".join(parts)

def build_eval_queries(items_df, max_queries=300):
    """Build a list of (query, ground_truth_index) for evaluation."""
    queries = []
    for i, row in items_df.iterrows():
        q = build_query(row)
        if q and len(q) >= 3:
            queries.append((q, i))
    random.shuffle(queries)
    return queries[:max_queries]

def precision_at_k(ranked_list, ground_truth, k):
    """Precision@K: 1/K if GT is in top-K, else 0."""
    return (1.0 / k) if ground_truth in ranked_list[:k] else 0.0

def recall_at_k(ranked_list, ground_truth, k):
    """Recall@K: 1 if GT is in top-K, else 0."""
    return 1.0 if ground_truth in ranked_list[:k] else 0.0

def f1_at_k(ranked_list, gt, k=10):
    """F1@K: harmonic mean of precision and recall."""
    p = precision_at_k(ranked_list, gt, k)
    r = recall_at_k(ranked_list, gt, k)
    return (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

def evaluate(queries, liked_indices, alpha, k=10):
    """Evaluate the model with Precision@K, Recall@K, and Latency."""
    total_prec = 0.0
    total_rec = 0.0
    total_time = 0.0
    total_f1 = 0.0

    for query_text, gt_index in queries:
        start = time.perf_counter()
        ranked = rank_cbf_ufw(query_text, liked_indices, alpha=alpha, topk=k)
        elapsed_ms = (time.perf_counter() - start) * 1000

        total_prec += precision_at_k(ranked, gt_index, k)
        total_rec += recall_at_k(ranked, gt_index, k)
        total_time += elapsed_ms
        total_f1 += f1_at_k(ranked, int(gt_index), k)

    n = max(1, len(queries))
    return {
        f"precision@{k}": total_prec / n,
        f"recall@{k}": total_rec / n,
        f"f1@{k}": total_f1 / n,
        f"latency_ms": total_time / n
    }

## 9) Running Evaluation & **UFW Justification**

In [19]:
# Build evaluation queries
EVAL_QUERIES = build_eval_queries(df, max_queries=300)
print(f"[Eval] Number of queries: {len(EVAL_QUERIES)}")

# Simulate liked items (take 3 items from the most frequent category)
top_categories = df["category"].fillna("").apply(lambda s: str(s).split(",")[0].strip())
dominant_cat = top_categories.value_counts().index[0] if len(top_categories.value_counts()) > 0 else ""
LIKED_IDX = df.index[top_categories == dominant_cat].tolist()[:3] if dominant_cat else []
print(f"[Eval] Pseudo-liked indices (category: '{dominant_cat}'): {LIKED_IDX}")

# Use fixed alpha = 0.6 (SIMPLIFIED - without tuning)
ALPHA = 0.6

# Run evaluation
metrics = evaluate(EVAL_QUERIES, LIKED_IDX, alpha=ALPHA, k=K_EVAL)

# Display results
results_df = pd.DataFrame([{
    "Method": f"CBF+UFW (α={ALPHA})",
    **metrics
}]).set_index("Method")

print("\n=== EVALUATION RESULTS ===")
display(results_df)

# Explanation
explanation = f"""
### ℹ️ Metric Explanation
- **Precision@{K_EVAL}**: Proportion of relevant items in Top-{K_EVAL} (1/{K_EVAL} if GT exists, 0 otherwise)
- **Recall@{K_EVAL}**: 1 if ground truth appears in Top-{K_EVAL}, 0 otherwise
- **Latency (ms)**: Average ranking time per query
- **F1@{K_EVAL}**: Harmonic mean of Precision@{K_EVAL} and Recall@{K_EVAL}

**Alpha = {ALPHA}** was chosen as the default value that provides a balance between query similarity and user preference.
"""
display(Markdown(explanation))

[Eval] Number of queries: 182
[Eval] Pseudo-liked indices (category: 'Cagar Alam'): [4, 5, 6]

=== EVALUATION RESULTS ===


,precision@10,recall@10,f1@10,latency_ms
Method,,,,
CBF+UFW (α=0.6),0.085714,0.857143,0.155844,4.675622



### ℹ️ Metric Explanation
- **Precision@10**: Proportion of relevant items in Top-10 (1/10 if GT exists, 0 otherwise)
- **Recall@10**: 1 if ground truth appears in Top-10, 0 otherwise
- **Latency (ms)**: Average ranking time per query
- **F1@10**: Harmonic mean of Precision@10 and Recall@10

**Alpha = 0.6** was chosen as the default value that provides a balance between query similarity and user preference.
